# Ninai Adapter Smoke Tests

Verifies that all three framework adapters (LangChain, Google ADK, Ninai SDK core)
are importable, structurally correct, and execute without errors.

Run this notebook after any SDK or adapter change as a sanity check.

In [1]:
import sys, os, warnings
warnings.filterwarnings('ignore')
SDK_PATH = os.path.abspath(os.path.join(os.getcwd(), '..', '..', 'sdk', 'python'))
if SDK_PATH not in sys.path:
    sys.path.insert(0, SDK_PATH)

import langchain_core
import google.adk
import ninai
from ninai import NinaiClient

print(f'langchain_core : {langchain_core.__version__}')
print(f'google-adk     : {google.adk.__version__}')
print(f'ninai SDK      : {ninai.__version__}')
print()
print('All framework imports OK.')


langchain_core : 1.2.28
google-adk     : 1.29.0
ninai SDK      : 1.0.0

All framework imports OK.


## Shared mock client

In [2]:
from unittest.mock import MagicMock
from types import SimpleNamespace

_STORE: dict = {}
_CTR = [0]

def _mock_create(**kwargs):
    _CTR[0] += 1
    m = SimpleNamespace(id=str(_CTR[0]), content=kwargs.get('content', ''),
                        title=kwargs.get('title', ''), tags=kwargs.get('tags', []))
    _STORE[m.id] = m
    return m

def _mock_search(query, **kwargs):
    items = [SimpleNamespace(memory_id=m.id, content=m.content, score=0.9, title=m.title)
             for m in list(_STORE.values())[-5:]]
    return SimpleNamespace(items=items[:3], total=len(items))

client = MagicMock()
client.memories.create.side_effect = _mock_create
client.memories.search.side_effect = _mock_search
print('Shared mock client ready')


Shared mock client ready


## LangChain adapter

In [3]:
# --- LangChain adapter smoke ---
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_core.messages import HumanMessage, AIMessage
from langchain_core.tools import BaseTool
from typing import List, Sequence, Optional, Type
from pydantic import BaseModel, Field
from langchain_core.callbacks import CallbackManagerForToolRun


class NinaiChatMessageHistory(BaseChatMessageHistory):
    def __init__(self, session_id, ninai_client):
        self.session_id = session_id
        self._client = ninai_client
        self._messages: List = []
    @property
    def messages(self): return list(self._messages)
    def add_messages(self, messages):
        for m in messages:
            self._client.memories.create(content=m.content, title='lc', tags=[])
            self._messages.append(m)
    def clear(self): self._messages.clear()

class _SI(BaseModel):
    query: str
    top_k: int = 5

class NinaiSearchTool(BaseTool):
    name: str = 'ninai_search'
    description: str = 'Search Ninai memory.'
    args_schema: Type[BaseModel] = _SI
    ninai_client: object = None
    class Config: arbitrary_types_allowed = True
    def _run(self, query, top_k=5, run_manager=None):
        res = self.ninai_client.memories.search(query)
        return '\n'.join(r.content for r in res.items) or 'none'

# Assertions
hist = NinaiChatMessageHistory('s1', client)
hist.add_messages([HumanMessage(content='hello'), AIMessage(content='hi')])
assert len(hist.messages) == 2

tool = NinaiSearchTool(ninai_client=client)
out = tool._run('hello')
assert isinstance(out, str)

print('LangChain adapter smoke: PASS')


LangChain adapter smoke: PASS


## Google ADK adapter

In [4]:
# --- ADK adapter smoke ---
from google.adk.tools import FunctionTool


def ninai_search_memory(query: str, top_k: int = 5) -> dict:
    """Search Ninai memory.

    Args:
        query: Search query string.
        top_k: Maximum results.

    Returns:
        dict with results and count.
    """
    res = client.memories.search(query, limit=top_k)
    return {'results': [r.content for r in res.items], 'count': len(res.items)}


def ninai_store_memory(content: str, tags: str = '') -> dict:
    """Store a memory in Ninai.

    Args:
        content: Content to store.
        tags: Comma-separated tags.

    Returns:
        dict with memory_id and status.
    """
    tag_list = [t.strip() for t in tags.split(',') if t.strip()]
    mem = client.memories.create(content=content, tags=tag_list)
    return {'memory_id': mem.id, 'status': 'stored'}


search_ft = FunctionTool(ninai_search_memory)
store_ft = FunctionTool(ninai_store_memory)

# Assertions
res = ninai_store_memory('test fact for ADK smoke', tags='test')
assert res['status'] == 'stored'

hits = ninai_search_memory('test fact')
assert isinstance(hits['results'], list)

assert search_ft.name == 'ninai_search_memory'
assert store_ft.name == 'ninai_store_memory'

print('ADK adapter smoke: PASS')


ADK adapter smoke: PASS


## Ninai SDK core

In [5]:
# --- SDK core smoke ---
from ninai import (
    NinaiClient, GoalPlannerAgent, GoalLinkingAgent,
    MetaAgent, ToolInvoker, InMemoryEventSink,
)

# Structural checks
assert issubclass(NinaiClient, object)
assert hasattr(NinaiClient, '__init__')

from types import SimpleNamespace
m = SimpleNamespace(id='x', content='test', title='t', tags=[])
assert m.id == 'x'
assert m.content == 'test'

sink = InMemoryEventSink()
assert hasattr(sink, 'events')

print('Ninai SDK core smoke: PASS')


Ninai SDK core smoke: PASS


## Summary

In [6]:
print('=' * 40)
print('All adapter smoke tests PASSED')
print('  LangChain adapter : OK')
print('  Google ADK adapter : OK')
print('  Ninai SDK core     : OK')
print('=' * 40)


All adapter smoke tests PASSED
  LangChain adapter : OK
  Google ADK adapter : OK
  Ninai SDK core     : OK
